## Auto Loader: Online Source to Delta Table

This notebook demonstrates how to:
1. **Create a Unity Catalog** and **schema** for the data pipeline
2. **Create a UC Volume** to stage files downloaded from an online source
3. **Download a public CSV dataset** from the internet into the volume
4. **Use Auto Loader** (`cloudFiles`) to incrementally load the staged files into a **Delta table**
5. **Query and display** the loaded data

**Online source:** Titanic dataset (CSV) from `raw.githubusercontent.com`

In [0]:
# ============================================================
# Cell 1: Create Unity Catalog and Schema
# ============================================================

# Configuration — change these to match your naming conventions
catalog_name = "autoloader_demo"
schema_name  = "bronze"

# Create the catalog (skip if it already exists)
print(f"Creating catalog '{catalog_name}'...")
spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS {catalog_name}
    COMMENT 'Auto Loader demo catalog'
""")

# Create the schema inside the catalog
print(f"Creating schema '{catalog_name}.{schema_name}'...")
spark.sql(f"""
    CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
    COMMENT 'Bronze layer for raw ingested data'
""")

# Verify
spark.sql(f"SHOW CATALOGS").filter(f"catalog == '{catalog_name}'").display()
spark.sql(f"SHOW SCHEMAS IN {catalog_name}").filter(f"databaseName == '{schema_name}'").display()
print(f"✅ Catalog '{catalog_name}' and schema '{catalog_name}.{schema_name}' are ready.")

In [0]:
# ============================================================
# Cell 2: Create UC Volume & Download Dataset from Online Source
# ============================================================
import urllib.request
import os

volume_name   = "staging_files"
full_volume   = f"{catalog_name}.{schema_name}.{volume_name}"
volume_path   = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

table_name    = "titanic_passengers"
full_table    = f"{catalog_name}.{schema_name}.{table_name}"

# Create the UC volume (managed)
print(f"Creating volume '{full_volume}'...")
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {full_volume}
    COMMENT 'Staging area for files downloaded from online sources'
""")

# --- Download the Titanic CSV directly into the UC volume ---
source_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
dest_path = f"{volume_path}/titanic.csv"

print(f"Downloading from: {source_url}")
print(f"Writing to: {dest_path}")
with urllib.request.urlopen(source_url) as response:
    data = response.read()
with open(dest_path, "wb") as f:
    f.write(data)
print(f"✅ File staged at: {dest_path} ({len(data)} bytes)")

# List the volume contents to confirm
print("\nFiles in volume:")
for item in dbutils.fs.ls(volume_path):
    print(f"  {item.name}  ({item.size} bytes)")

In [0]:
# ============================================================
# Cell 3: Auto Loader — Incrementally Read Staged Files into Delta Table
# ============================================================
# Auto Loader uses the `cloudFiles` format to automatically discover
# new files in the source directory and incrementally process them.

# Schema location: Auto Loader stores the inferred/evolved schema here
schema_location = f"{volume_path}/_schemas/titanic"

# Checkpoint location: Auto Loader tracks processed files here
checkpoint_location = f"{volume_path}/_checkpoints/titanic"

print("Starting Auto Loader stream...")
print(f"  Source:            {volume_path}")
print(f"  Schema location:   {schema_location}")
print(f"  Checkpoint location: {checkpoint_location}")
print(f"  Target table:      {full_table}")

# Drop the table if it already exists (for re-runs)
spark.sql(f"DROP TABLE IF EXISTS {full_table}")

# Configure and start the Auto Loader stream
query = (
    spark.readStream
    .format("cloudFiles")
    # cloudFiles.format: Tells Auto Loader what file format to expect. Use when your source directory
    # contains a single file type (e.g. csv, json, parquet, text, binaryFile).
    .option("cloudFiles.format", "csv")
    # cloudFiles.schemaLocation: Directory where Auto Loader stores the inferred/evolved schema.
    # Use so that schema is remembered across stream restarts and new columns are tracked over time.
    .option("cloudFiles.schemaLocation", schema_location)
    # cloudFiles.schemaEvolutionMode: Controls how schema changes are handled. "addNewColumns"
    # automatically adds newly discovered columns to the schema. Use when source files may evolve
    # and gain columns over time without breaking the stream.
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    # header: Indicates the first row of each CSV file contains column names. Use with CSV sources
    # that include a header row so column names are parsed correctly.
    .option("header", "true")
    # inferSchema: Lets Auto Loader infer column data types (int, string, etc.) from a sample
    # of the data. Use during initial exploration; for production, provide an explicit schema
    # via .schema() or a schema file for consistency and performance.
    .option("inferSchema", "true")
    .load(volume_path)
    .writeStream
    .format("delta")
    # checkpointLocation: Directory where the streaming engine tracks progress (which files have
    # been processed). Use so the stream can resume after a restart without reprocessing or losing data.
    .option("checkpointLocation", checkpoint_location)
    # mergeSchema: Allows the Delta table schema to evolve by merging new columns added by the stream
    # into the target table. Use together with schemaEvolutionMode so inferred new columns are written
    # to the Delta table without manual ALTER TABLE statements.
    .option("mergeSchema", "true")
    # trigger(availableNow=True): Processes ALL files that are currently available and then stops the
    # stream. Use for batch-style incremental ingestion (run on a schedule) instead of continuous running.
    .trigger(availableNow=True)  # Process all available files, then stop
    .toTable(full_table)
)

# Wait for the stream to finish processing all available files
query.awaitTermination()
print(f"✅ Auto Loader stream completed. Data loaded into {full_table}")

In [0]:
# ============================================================
# Cell 4: Verify and Display the Loaded Data
# ============================================================

# Show the schema of the loaded table
print("Table schema:")
spark.sql(f"DESCRIBE TABLE {full_table}").display()

# Count rows
row_count = spark.sql(f"SELECT COUNT(*) AS total_rows FROM {full_table}").collect()[0]["total_rows"]
print(f"\nTotal rows loaded: {row_count}")

# Display sample data
print(f"\nSample data from {full_table}:")
spark.sql(f"SELECT * FROM {full_table} LIMIT 10").display()

# Quick summary by survival
print("\nSurvival summary:")
spark.sql(f"""
    SELECT 
        Survived,
        COUNT(*) AS passenger_count,
        ROUND(AVG(Age), 1) AS avg_age,
        ROUND(AVG(Fare), 2) AS avg_fare
    FROM {full_table}
    GROUP BY Survived
    ORDER BY Survived
""").display()

In [0]:
# ============================================================
# Cell 5 (Optional): Cleanup — Drop the table, volume, schema, and catalog
# ============================================================
# Uncomment the lines below if you want to remove all created resources.

# spark.sql(f"DROP TABLE IF EXISTS {full_table}")
# spark.sql(f"DROP VOLUME IF EXISTS {full_volume}")
# spark.sql(f"DROP SCHEMA IF EXISTS {catalog_name}.{schema_name}")
# spark.sql(f"DROP CATALOG IF EXISTS {catalog_name}")
# print("✅ All resources cleaned up.")